In [1]:
import json
from openai import OpenAI

# Initialize the OpenAI client
client = OpenAI(base_url="http://172.30.80.1:7878/v1", api_key="lm-studio")

In [24]:
prompt_placeholder = """
{story}
** Objective: Extract key insights from the customer testimonial about [Company Name]'s Business Intelligence (BI) adoption and implementation process. **

Guiding Questions:
* Adoption Drivers: What were the primary drivers behind [Company Name]'s BI adoption? (e.g., [adoption driver 1], [adoption driver 2], etc.)
* Implementation Strategies: Which strategies did [Company Name] employ to successfully deploy its BI capabilities? (e.g., [implementation strategy 1], [implementation strategy 2], etc.)
* Challenges and Solutions: What were the most common challenges faced by [Company Name] during its BI implementation process, and how were they addressed? (e.g., [common challenge 1], [common challenge 2], etc.)
* Monetization: How did [Company Name] monetize its BI capabilities? (e.g., [monetization approach 1], [monetization approach 2], etc.)
* Collaboration and Culture: What strategies did [Company Name] use to promote team collaboration and data-driven decision-making within its organization as a result of implementing BI?
* Mobile Accessibility: Does [Company Name] display their dashboard in mobile apps?
* Ease of Use: How intuitive and user-friendly is the platform? Consider feedback related to the learning curve and user interface.
* Functionality: What features are available (e.g., data visualization, report generation, analytics)? Assess whether the platform meets the diverse needs of users.
* Integration Capabilities: How easily can the platform integrate with other tools and systems in the users’ environments?
* Scalability: Can the platform scale according to the organization’s needs? Consider feedback on its performance as the amount of data grows.
* Cost Effectiveness: Assess perceptions regarding the platform’s pricing relative to its features and capabilities.
* Customer Support: How effective is the customer support? Review comments about the responsiveness and helpfulness of the support team.
* Data Security: How well does the platform secure user data? Include feedback on any security features and users’ trust in the platform.
* Customization Options: How adaptable is the platform to specific business needs? Consider how users feel about the customization capabilities.
* Training and Resources: Review the availability and quality of training materials and resources to help users maximize their use of the platform.

**Instructions to the Model:**
1. **Prioritize Accuracy:** Extract information directly from the text. Do not fabricate or "hallucinate" details.
2. **Handle Missing Information:** If a topic is not mentioned in the testimonial, indicate "Information not found" in the corresponding JSON field.
3. **Output Format:** Provide a JSON object with the following keys: `drivers`, `imp_strategies`, `challenges`, `monetization`, `collaboration`, `mobile`, `ease_of_use`,  `functionality`, `integration`, `scalability`, `cost_effectiveness`, `customer_support`, `data_security`, `customization`, 'training'.

"""

In [50]:
# Load the JSON file with stories
with open('stories.json') as f:
    stories = json.load(f)

def collect_data(stories, prompt_placeholder, client):
    """Collects data from the LLM model, storing results in an intermediate format.

    Args:
        stories: A list of stories (each as a dictionary with a 'content' key).
        prompt_placeholder: The prompt template with a '{story}' placeholder.
        client: The client object for interacting with the LLM.

    Returns:
        A list of results where each result is a dictionary.
    """

    results = []
    for i, story in enumerate(stories):
        content = story['content']
        story_id = str(i)
        prompt = prompt_placeholder.replace('{story}', content)

        completion = client.chat.completions.create(
            model="MaziyarPanahi/Meta-Llama-3-8B-Instruct-GGUF",
            messages=[
                {"role": "system", "content": "You are a helpful, smart, and efficient AI assistant, you're task is to analyze a customer testimonial from a BI service and extract some variables from it."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.2,
        )

        response_text = completion.choices[0].message.content.replace('\n', '')

        results.append({
            'story_id': story_id,
            # 'story_content': content,  # Include the original content
            'response': response_text,
        })

    return results

def convert_to_json(results):
    """Converts the collected results into JSON format.

    Args:
        results: The list of results produced by the 'collect_data' function.

    Returns:
        A list of results where each result is a dictionary with extracted JSON data.
    """

    json_results = []
    for result in results:
        json_start_index = result['response'].find('{')
        json_end_index = result['response'].rfind('}') + 1
        json_text = result['response'][json_start_index:json_end_index]

        try:
            extracted_data = json.loads(json_text)
            json_results.append({
                'story_id': result['story_id'],
                # 'story_content': result['story_content'],
                **extracted_data
            })
        except json.JSONDecodeError:
            json_results.append({
                'story_id': result['story_id'],
                # 'story_content': result['story_content'],
                'json_error': 'Invalid JSON in response'
            })

    return json_results

In [29]:
data = collect_data(stories, prompt_placeholder, client)  # Step 1
json_data = convert_to_json(data)         # Step 2

with open('results.json', 'w') as f:
    json.dump(json_data, f)  

In [30]:
# Load the JSON file with stories
with open('results.json') as f:
    content = json.load(f)

# Define a placeholder for your prompt (replace `{story}` with the actual story)
prompt_placeholder = """
{story}
The following is the results of analysing 10 customer testimonials from BI services. There are 5 variables extracted from each testimonial. The variables are:
* Adoption Drivers: What were the primary drivers behind [Company Name]'s BI adoption? (e.g., [adoption driver 1], [adoption driver 2], etc.)
* Implementation Strategies: Which strategies did [Company Name] employ to successfully deploy its BI capabilities? (e.g., [implementation strategy 1], [implementation strategy 2], etc.)
* Challenges and Solutions: What were the most common challenges faced by [Company Name] during its BI implementation process, and how were they addressed? (e.g., [common challenge 1], [common challenge 2], etc.)
* Monetization: How did [Company Name] monetize its BI capabilities? (e.g., [monetization approach 1], [monetization approach 2], etc.)
* Collaboration and Culture: What strategies did [Company Name] use to promote team collaboration and data-driven decision-making within its organization as a result of implementing BI?
* Mobile Accessibility: Does [Company Name] display their dashboard in mobile apps?
* Ease of Use: How intuitive and user-friendly is the platform? Consider feedback related to the learning curve and user interface.
* Functionality: What features are available (e.g., data visualization, report generation, analytics)? Assess whether the platform meets the diverse needs of users.
* Performance: How well does the platform perform in terms of speed and efficiency, especially when handling large datasets?
* Integration Capabilities: How easily can the platform integrate with other tools and systems in the users’ environments?
* Scalability: Can the platform scale according to the organization’s needs? Consider feedback on its performance as the amount of data grows.
* Cost Effectiveness: Assess perceptions regarding the platform’s pricing relative to its features and capabilities.
* Customer Support: How effective is the customer support? Review comments about the responsiveness and helpfulness of the support team.
* Data Security: How well does the platform secure user data? Include feedback on any security features and users’ trust in the platform.
* Customization Options: How adaptable is the platform to specific business needs? Consider how users feel about the customization capabilities.
* Accuracy and Reliability: Evaluate feedback concerning the accuracy and reliability of the data analysis provided by the platform.
* Training and Resources: Review the availability and quality of training materials and resources to help users maximize their use of the platform.

**Analyse all variables and output a conclusion to this study, including some numbers and statistics about each variable. Try to categories answers of variables into groups and provide a conclusion.**
"""

# Generate the prompt for the current story
prompt = prompt_placeholder.replace('{story}', str(content))

completion = client.chat.completions.create(
    model="MaziyarPanahi/Meta-Llama-3-8B-Instruct-GGUF",
    messages=[
        {"role": "system", "content": "You are a helpful, smart, and efficient AI assistant, you're task is to analyze a customer testimonial from a BI service and extract some variables from it."},
        {"role": "user", "content": prompt}
    ],
    temperature=0.2,
)

# Extract JSON data from the response 
response_text = completion.choices[0].message

In [31]:
print(response_text.content)

**Conclusion**

The analysis of 10 customer testimonials from BI services reveals a comprehensive picture of the key factors that contribute to successful business intelligence implementations. The results are categorized into the following variables:

1. **Adoption Drivers**: The primary drivers behind companies' BI adoption were:
	* Personal contact and communication with financial institutions (40%)
	* Importance of relationships with customers, members, and employees (30%)
	* Other factors (30%)
2. **Implementation Strategies**: Companies employed various strategies to successfully deploy their BI capabilities, including:
	* Structured online questionnaires (60%)
	* Tableau adoption for data analysis (50%)
	* Other strategies (40%)
3. **Challenges and Solutions**: The most common challenges faced by companies during their BI implementation process were:
	* Evaluation time (70%)
	* Need for professionalized analysis (60%)
	* Other challenges (30%)
4. **Monetization**: Companies mone

In [33]:
## Deep Insights

In [43]:
prompt_placeholder_deep = """
{story}
**Objective:** Extract key insights from the customer testimonial about [Company Name]'s Business Intelligence (BI) adoption and implementation process.

**Guiding Questions:**
1. **Feedback Integration:** How is customer feedback, through direct input or satisfaction surveys, integrated and analyzed within the BI platform to continually improve the training experience?
2. **Social Responsibility Support:** How does the BI platform assist in promoting social responsibility, including features or analyses related to community engagement, diversity and inclusion, and ethical business practices?
3. **Data Balance (Secondary Datasets):** Identify and categorize any mentions of external data, secondary data, or data sources beyond the company's primary dataset. How is the BI tool used to merge, blend, or relate internal and external datasets?
4. **Dynamic Stakeholder Interests:** How do users discuss responding to changes in business questions, evolving stakeholder requests, or the need for new visualizations?
5. **Societal Impact:** How does the platform demonstrate societal impact to stakeholders, including public funders?
6. **Strategic Planning:** How does the platform help in building evidence to support bidding for future contracts?
7. **Targeting Interventions/Campaigns:** How does the platform aid in building evidence for targeting interventions or campaigns?
8. **Business Expansion:** How does the platform enable the company to expand its offering from service to service + information/insight (vertical integration)?
9. **Stakeholder Engagement:** What evidence is there of becoming a more attractive partner for clients/collaborators by being able to share insights or be transparent?
10. **Performance Tracking:** How does the BI system capture and analyze data on staff/users performance?
11. **Behavioral Change:** How does the BI platform contribute to facilitating behavioral change among users/clients?
12. **Financial Insights:** How does the platform provide insights into revenue and other financial metrics?

**Instructions to the Model:**
1. **Prioritize Accuracy:** Extract information directly from the text. Do not fabricate or "hallucinate" details.
2. **Handle Missing Information:** If a topic is not mentioned in the testimonial, indicate "Information not found" in the corresponding JSON field.

**Output:** Summarize the extracted information into JSON format with keys:
- `feedback_integration`
- `social_responsibility`
- `data_balance`
- `dynamic_stakeholders`
- `societal_impact`
- `strategic_planning`
- `targeting_interventions`
- `business_expansion`
- `stakeholder_engagement`
- `performance_tracking`
- `behavioral_change`
- `financial_insights`

"""

In [57]:
data_deep = collect_data(stories, prompt_placeholder_deep, client)  # Step 1
# json_data_deep = convert_to_json(data_deep)         # Step 2

with open('results_deep.json', 'w') as f:
    json.dump(data_deep, f)  